In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F



class ResBlock(nn.Module):
  def __init__(self):
    super().__init__()
    self.bn1=nn.BatchNorm2d(128)

    self.relu=nn.ReLU()
    self.conv1=nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)

    self.bn2=nn.BatchNorm2d(128)
    self.relu2=nn.ReLU()
    self.conv2=nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)

  def forward(self, x):
    residual=x
    x=self.bn1(x)
    x=self.relu(x)
    x=self.conv1(x)

    x=self.bn2(x)
    x=self.relu2(x)
    x=self.conv2(x)


    x+=residual
    return x

class ResBlock_discriminator(nn.Module):
  def __init__(self, in_channels, out_channels, first_resblock=False, downsample=True):
    super().__init__()
    if first_resblock==False:
      self.relu=nn.ReLU()
    else:
      self.relu=nn.Identity()
    self.conv1=nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)


    self.relu2=nn.ReLU()

    self.conv2=nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
    if downsample==True:
      self.avg_pool=nn.AvgPool2d(2)
    else:
      self.avg_pool=nn.Identity()
    if in_channels!=out_channels or downsample:
      self.conv3=nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
  def forward(self, x):
    residual=x
    x=self.relu(x)
    x=self.conv1(x)

    x=self.relu2(x)

    x=self.conv2(x)
    x=self.avg_pool(x)
    residual=self.avg_pool(residual)
    if hasattr(self, 'conv3'):
      residual=self.conv3(residual)
    x+=residual

    return x
class Generator(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear1=nn.Linear(128, 2048)

    self.resblock1=ResBlock()
    self.resblock2=ResBlock()
    self.resblock3=ResBlock()

    self.bn_final=nn.BatchNorm2d(128)
    self.relu=nn.ReLU()
    self.conv=nn.Conv2d(128, 3, kernel_size=3, stride=1, padding=1)
    self.tanh=nn.Tanh()

  def forward(self, x):
    x=self.linear1(x)
    x=x.reshape(-1, 128, 4,4)
    x=F.interpolate(x, scale_factor=2, mode='nearest')
    x=self.resblock1(x)
    x=F.interpolate(x, scale_factor=2, mode='nearest')
    x=self.resblock2(x)
    x=F.interpolate(x, scale_factor=2, mode='nearest')
    x=self.resblock3(x)

    x=self.bn_final(x)
    x=self.relu(x)
    x=self.conv(x)
    x=self.tanh(x)
    return x

class Discriminator(nn.Module):
  def __init__(self):
    super().__init__()
    self.resblock1=ResBlock_discriminator(3, 128, True, True)
    self.resblock2=ResBlock_discriminator(128, 128, False, True)
    self.resblock3=ResBlock_discriminator(128, 128, False, False)
    self.resblock4=ResBlock_discriminator(128, 128, False, False)

    self.relu=nn.ReLU()
    self.mean_pool=nn.AvgPool2d(8)
    self.linear=nn.Linear(128, 1)

  def forward(self, x):
    x=self.resblock1(x)
    x=self.resblock2(x)
    x=self.resblock3(x)
    x=self.resblock4(x)

    x=self.relu(x)
    x=self.mean_pool(x)
    x=x.flatten(1)
    x=self.linear(x)
    return x



In [ ]:
class Wasserstein_GAN(nn.Module):
  def __init__(self):
    self.generator=Generator()
    self.discriminator=Discriminator()
  def forward(self, x):
    x=self.generator(x)
    x=self.discriminator(x)
    return x


In [ ]:
discriminator=Discriminator()
generator=Generator()

In [ ]:
def loss(true_logits, pred_logits,  lam,  true_interpolated_image, interpolated_logits):

  loss=pred_logits-true_logits+lam * (torch.linalg.norm(torch.autograd.grad(interpolated_logits,true_interpolated_image, create_graph=True)[0], ord=2)-1)**2
  return loss

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision import transforms
training_data=datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

eval_data=datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

train_set=torch.utils.data.DataLoader(training_data, batch_size=64, shuffle=True)
eval_set=torch.utils.data.DataLoader(eval_data, batch_size=64, shuffle=False)


lr=2e-4
#linear decay in Adam from 0 to 100k generator iterations

optimizer_discriminator=torch.optim.Adam(discriminator.parameters(), lr, (0, 0.9))

optimizer_generator=torch.optim.Adam(generator.parameters(), lr, (0, 0.9))

100%|██████████| 170M/170M [23:00<00:00, 124kB/s]


ValueError: betas must be either both floats or both Tensors

In [ ]:
optimizer_discriminator=torch.optim.Adam(discriminator.parameters(), lr, (0.0, 0.9))

optimizer_generator=torch.optim.Adam(generator.parameters(), lr, (0.0, 0.9))

In [ ]:
def loss(true_logits, pred_logits,  lam,  true_interpolated_image, interpolated_logits):

  gradient=torch.autograd.grad(interpolated_logits, true_interpolated_image, torch.ones_like(interpolated_logits), create_graph=True, retain_graph=True)[0]
  gradient=gradient.reshape(gradient.size(0),-1)
  gradient_norm=gradient.norm(2, dim=1)
  loss=pred_logits.mean()-true_logits.mean()+lam * ((gradient_norm-1)**2).mean()
  return loss

In [ ]:
latent=torch.rand(64, 128)
real_images=torch.rand(64,3,32,32)
print(real_images.shape)
generated_images=generator(latent)
print(generated_images.shape)
logits=discriminator(generated_images)
interpolant=torch.rand(1)
interpolated_fake=(interpolant*(real_images/255.0)+(1-interpolant)*generated_images)
print(interpolated_fake.shape)
print(logits.shape, discriminator(real_images).shape, interpolated_fake.shape, discriminator(interpolated_fake).shape)
loss_result=loss(logits, discriminator(real_images), 10, interpolated_fake, discriminator(interpolated_fake))
print(loss_result)

torch.Size([64, 3, 32, 32])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3, 32, 32])
torch.Size([64, 1]) torch.Size([64, 1]) torch.Size([64, 3, 32, 32]) torch.Size([64, 1])
tensor(35.1688, grad_fn=<AddBackward0>)


In [ ]:
single_batch=next(iter(train_set))
images=single_batch[0]
print(images.min())

tensor(0.)


In [ ]:
# one train loop over batch

for single_batch in train_set:

  images=single_batch[0]

  loss_generator=0
  for i in range(5):
    loss_discriminator=0

    latent=torch.rand(64, 128)

    interpolant=torch.rand(images.size(0),1,1,1)
    with torch.no_grad():
      fake_images=generator(latent)

    interpolated_fakes=(interpolant*(images)+(1-interpolant)*fake_images).requires_grad_(True)
    loss_discriminator+=loss( discriminator(images),discriminator(fake_images), 10, interpolated_fakes, discriminator(interpolated_fakes))




    loss_discriminator=loss_discriminator

    optimizer_discriminator.zero_grad()

    loss_discriminator.backward()

    optimizer_discriminator.step()
  for p in discriminator.parameters():
    p.requires_grad_(False)
  latent2=torch.rand(64,128)
  fake_images2=generator(latent2)
  loss_generator=(-discriminator(fake_images2).mean())
  optimizer_generator.zero_grad()
  loss_generator.backward()
  optimizer_generator.step()
  for p in discriminator.parameters():
    p.requires_grad_(True)
  print("loss generator: ", loss_generator.item())

loss generator:  0.7852447628974915
loss generator:  -27.403650283813477
loss generator:  56.53528594970703
loss generator:  28.23387336730957
loss generator:  39.435523986816406
loss generator:  52.37892532348633
loss generator:  58.26072311401367
loss generator:  17.20895767211914
loss generator:  71.82212829589844
loss generator:  46.12308120727539
loss generator:  0.21352213621139526
loss generator:  79.49551391601562
loss generator:  60.16717529296875
loss generator:  2.1563563346862793
loss generator:  79.81478881835938
